In [12]:
# Install required libraries
%pip install shap joblib
%pip install mlflow
%pip install catboost
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.4 MB/s  0:00:0036m-:--:--
Note: you may need to restart the kernel to use updated packages.


In [15]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

from scipy.io import loadmat
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score, cohen_kappa_score,
    log_loss, average_precision_score, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
# from lightgbm import LGBMClassifier

import mlflow
import mlflow.sklearn
import joblib
from sklearn.preprocessing import label_binarize

In [2]:
# Load Data
elec_data = loadmat('/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/datasets/dataset_elec.mat')
amb_data = loadmat('/Users/seyedrumaiz/Library/CloudStorage/OneDrive-InformaticsInstituteofTechnology/DSGP/solar-panel-fault-mapping/datasets/dataset_amb.mat')

vdc1 = elec_data['vdc1'].flatten()
vdc2 = elec_data['vdc2'].flatten()
idc1 = elec_data['idc1'].flatten()
idc2 = elec_data['idc2'].flatten()

irr = amb_data['irr'].flatten()
pvt = amb_data['pvt'].flatten()
f_nv = amb_data['f_nv'].flatten()

# Create DataFrame
df = pd.DataFrame({
    'vdc1': vdc1,
    'vdc2': vdc2,
    'idc1': idc1,
    'idc2': idc2,
    'irradiance': irr,
    'temperature': pvt,
    'fault_label': f_nv
})

In [3]:
# Filter out unwanted labels
df = df[df['fault_label'] != 2].copy()

# Feature Engineering
df['power_string1'] = df['vdc1'] * df['idc1']
df['power_string2'] = df['vdc2'] * df['idc2']
df['total_power'] = df['power_string1'] + df['power_string2']
df['voltage_ratio'] = df['vdc1'] / df['vdc2']
df['current_ratio'] = df['idc1'] / df['idc2']

# Map fault names
fault_names = {0: 'Normal Operation', 1: 'Short-Circuit', 3: 'Open Circuit', 4: 'Shadowing'}
df['fault_label'] = df['fault_label'].map(fault_names)

In [4]:
# Prepare Features and Labels
X = df.drop(['fault_label'], axis=1)
y = df['fault_label']

In [5]:
# One-hot encode labels for multiclass metrics
ohe = OneHotEncoder()
y_encoded = ohe.fit_transform(y.values.reshape(-1, 1)).toarray()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
# Compute class weights to handle imbalance
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

Class weights: {'Normal Operation': np.float64(0.2931015301866836), 'Open Circuit': np.float64(56.585443037974684), 'Shadowing': np.float64(1.808509474131013), 'Short-Circuit': np.float64(56.82126484684309)}


In [7]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
def train_and_evaluate_model(
    model,
    model_name,
    X_train,
    X_test,
    y_train,
    y_test,
    experiment_name="PV_Fault_Detection_Base_Models"
):
    """
    Trains a sklearn model, evaluates it using robust multiclass metrics,
    visualizes ROC & PR curves, logs everything to MLflow.
    """

    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=model_name):

        # Train model
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)

        if not hasattr(model, "predict_proba"):
            raise ValueError(f"{model_name} does not support predict_proba()")

        y_proba = model.predict_proba(X_test)

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        kappa = cohen_kappa_score(y_test, y_pred)
        ll = log_loss(y_test, y_proba)

        # ROC and PR AUC
        y_test_bin = label_binarize(y_test, classes=model.classes_)

        roc_auc_macro = roc_auc_score(
            y_test_bin,
            y_proba,
            multi_class='ovr',
            average='macro'
        )

        pr_auc_macro = average_precision_score(
            y_test_bin,
            y_proba,
            average='macro'
        )

        # Document metrics
        metrics = {
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "precision_macro": prec,
            "recall_macro": rec,
            "f1_macro": f1,
            "cohen_kappa": kappa,
            "log_loss": ll,
            "roc_auc_macro": roc_auc_macro,
            "pr_auc_macro": pr_auc_macro
        }

        for k, v in metrics.items():
            mlflow.log_metric(k, v)

        # ROC AUC curves
        plt.figure(figsize=(8, 6))

        for i, class_name in enumerate(model.classes_):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
            auc_score = auc(fpr, tpr)

            plt.plot(fpr, tpr, lw=2,
                     label=f"{class_name} (AUC={auc_score:.3f})")

        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC–AUC Curve ({model_name})")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "roc_auc_curve.png")
        plt.close()

        # PR Curves
        plt.figure(figsize=(8, 6))

        for i, class_name in enumerate(model.classes_):
            precision, recall, _ = precision_recall_curve(
                y_test_bin[:, i],
                y_proba[:, i]
            )

            plt.plot(
                recall,
                precision,
                lw=2,
                label=f"{class_name}"
            )

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision–Recall Curve ({model_name})")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "precision_recall_curve.png")
        plt.close()

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
        disp = ConfusionMatrixDisplay(cm, display_labels=model.classes_)
        disp.plot(cmap='Blues')
        plt.title(f"Confusion Matrix ({model_name})")
        plt.tight_layout()

        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()

        # Metrics table
        metrics_df = pd.DataFrame.from_dict(
            metrics, orient='index', columns=['Value']
        )

        metrics_path = "metrics_summary.csv"
        metrics_df.to_csv(metrics_path)
        mlflow.log_artifact(metrics_path)

        # Log model and display output
        mlflow.sklearn.log_model(model, name="model")

        print(f"\n ~ {model_name}")
        print(metrics_df)
        print("\nClassification Report:\n")
        print(classification_report(y_test, y_pred))

In [15]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight=class_weights,
    n_jobs=-1
)

train_and_evaluate_model(
    rf_model,
    model_name="RandomForest_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/09 23:32:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/09 23:32:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/09 23:32:20 INFO mlflow.store.db.utils: Updating database tables
2026/02/09 23:32:20 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/09 23:32:20 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/09 23:32:20 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/09 23:3


 ~ RandomForest_Base
                      Value
accuracy           0.999179
balanced_accuracy  0.998045
precision_macro    0.998984
recall_macro       0.998045
f1_macro           0.998514
cohen_kappa        0.996757
log_loss           0.002351
roc_auc_macro      0.999997
pr_auc_macro       0.999984

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      1.00      1.00      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



The Random Forest model demonstrates excellent performance across all evaluation metrics. With an overall accuracy of 99.9%, the
model reliably predicts all fault types, even the less frequent ones. The per-class metrics show perfect/near-perfect precision,
recall, and F1-scores, indicating that the model correctly identifies all classes without misclassifying.

High values of Cohen's Kappa (0.997), log loss (0.002), ROC AUC (almost 1), and PR-AUC (almost 1) further confirm the model's
robustness and its ability to distinguish between normal operation and different electrical faults accurately.

In [ ]:
# svc = SVC(kernel='linear', probability=True, class_weight=class_weights, random_state=42)

# train_and_evaluate_model(
#     model=svc,
#     model_name="SVC_Base",
#     X_train=X_train,
#     X_test=X_test,
#     y_train=y_train,
#     y_test=y_test
# )

2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/09 23:57:57 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/09 23:57:57 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/09 23:57:57 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [9]:
nb = GaussianNB()

train_and_evaluate_model(
    model=nb,
    model_name="NaiveBayes_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/10 00:05:07 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/10 00:05:08 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/10 00:05:08 INFO alembic.runtime.migration: Will assume non-transactional DDL.



 ~ NaiveBayes_Base
                      Value
accuracy           0.806147
balanced_accuracy  0.915189
precision_macro    0.838967
recall_macro       0.915189
f1_macro           0.853217
cohen_kappa        0.472466
log_loss           0.851378
roc_auc_macro      0.962895
pr_auc_macro       0.902834

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       0.98      0.79      0.87    232587
    Open Circuit       0.98      1.00      0.99      1205
       Shadowing       0.41      0.88      0.56     37694
   Short-Circuit       0.99      0.99      0.99      1200

        accuracy                           0.81    272686
       macro avg       0.84      0.92      0.85    272686
    weighted avg       0.90      0.81      0.83    272686



In [29]:
mlflow.set_tracking_uri("file:///Users/seyedrumaiz/mlflow_local")

In [30]:
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1, weights='uniform')

train_and_evaluate_model(
    model=knn,
    model_name="KNN_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/10 00:45:36 INFO mlflow.tracking.fluent: Experiment with name 'PV_Fault_Detection_Base_Models' does not exist. Creating a new experiment.



 ~ KNN_Base
                      Value
accuracy           0.998771
balanced_accuracy  0.996858
precision_macro    0.997667
recall_macro       0.996858
f1_macro           0.997263
cohen_kappa        0.995150
log_loss           0.011963
roc_auc_macro      0.999593
pr_auc_macro       0.998626

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      0.99      0.99      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



In [ ]:
# Base estimator for AdaBoost
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)

# Initialize adaboost
adaboost = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=100,
    learning_rate=1.0,
    random_state=42,
    class_weights=class_weights
)

train_and_evaluate_model(
    model=adaboost,
    model_name="AdaBoost_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ AdaBoost_Base
                      Value
accuracy           0.864019
balanced_accuracy  0.644499
precision_macro    0.862442
recall_macro       0.644499
f1_macro           0.695748
cohen_kappa        0.197455
log_loss           1.315342
roc_auc_macro      0.926751
pr_auc_macro       0.901285

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       0.87      0.99      0.93    232587
    Open Circuit       1.00      0.59      0.74      1205
       Shadowing       0.59      0.10      0.17     37694
   Short-Circuit       0.99      0.90      0.94      1200

        accuracy                           0.86    272686
       macro avg       0.86      0.64      0.70    272686
    weighted avg       0.83      0.86      0.82    272686



In [ ]:
# sample_weights = np.array([class_weights[y]] for y in y_train)

# xgb_base = XGBClassifier(
#     objective="multi:softprob",
#     num_classes=len(np.unique(y_train)),
#     eval_metric="recall",
#     n_estimators=200,
#     learning_rate=0.1,
#     max_depth=6
# )

In [9]:
dt_base = DecisionTreeClassifier(
    class_weight=class_weights,
    random_state=42
)

train_and_evaluate_model(
    model=dt_base,
    model_name="DecisionTree_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)

2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/10 06:40:10 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/10 06:40:10 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/10 06:40:10 INFO alembic.runtime.migration: Will assume non-transactional DDL.



 ~ DecisionTree_Base
                      Value
accuracy           0.998881
balanced_accuracy  0.996634
precision_macro    0.996687
recall_macro       0.996634
f1_macro           0.996659
cohen_kappa        0.995583
log_loss           0.040315
roc_auc_macro      0.997727
pr_auc_macro       0.993614

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       1.00      1.00      1.00     37694
   Short-Circuit       1.00      0.99      0.99      1200

        accuracy                           1.00    272686
       macro avg       1.00      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686



In [11]:
cat = CatBoostClassifier(
    loss_function="MultiClass",
    auto_class_weights="Balanced",  # CatBoost will balance automatically
    verbose=0,
    random_state=42
)

train_and_evaluate_model(
    model=cat,
    model_name="CatBoost_Base",
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test
)


 ~ CatBoost_Base
                      Value
accuracy           0.997847
balanced_accuracy  0.998610
precision_macro    0.993829
recall_macro       0.998610
f1_macro           0.996205
cohen_kappa        0.991544
log_loss           0.005670
roc_auc_macro      0.999990
pr_auc_macro       0.999957

Classification Report:

                  precision    recall  f1-score   support

Normal Operation       1.00      1.00      1.00    232587
    Open Circuit       1.00      1.00      1.00      1205
       Shadowing       0.99      1.00      0.99     37694
   Short-Circuit       0.99      1.00      0.99      1200

        accuracy                           1.00    272686
       macro avg       0.99      1.00      1.00    272686
    weighted avg       1.00      1.00      1.00    272686

